# 경제 국면 자산배분 — MDD 유지형 CAGR 개선안

    **연구 질문:** 기존 검증 모델의 최대낙폭을 악화시키지 않으면서 장기 복리수익률을 얼마나 높일 수 있는가?

    기존 모델의 경제 국면 신호와 자산군은 그대로 유지하고, 새로운 레버리지나 자산을 추가하지 않았다. 채권 비중의 일부를 **추세가 확인된 기존 위험자산에만 최대 10%p 전술 배분**하며, 하락 추세·낙폭·변동성 조건에서는 자동으로 위험을 축소한다.

    > 결과는 연구용 역사적 시뮬레이션이며 미래 수익을 보장하지 않는다. 모든 수치는 거래·환전비용 차감 후이며 Sharpe의 무위험수익률은 0%다.

## 1. 실험 설계와 과적합 방지

    - **자산군 고정:** KODEX200, 국내 채권 총수익지수, GLD, USO.
    - **시점 규칙:** 발표시차를 반영한 `t월 말 신호 → t+1월 투자`; 미래 수익률은 신호에 사용하지 않는다.
    - **캘리브레이션:** 2007-04~2017-12에서만 후보를 선택한다.
    - **잠금 테스트:** 2018-01 이후는 선택 완료 후 한 번만 공개한다.
    - **엄격 관문:** 캘리브레이션 MDD가 기존 모델보다 조금이라도 나빠지거나 Sharpe가 1 미만이면 탈락한다.
    - **최종 추가 탐색:** 더 강한 방어를 붙인 15~20% 전술 비중 48개를 별도로 시험했으나 모두 엄격 MDD 관문에서 탈락했다.

    따라서 가장 높은 CAGR 숫자가 아니라, **MDD 비악화 조건을 실제로 통과한 10% 상한 구성**을 채택했다.

In [ ]:
from pathlib import Path
    import json
    from dataclasses import asdict

    import numpy as np
    import pandas as pd
    from IPython.display import display, Markdown
    import matplotlib.pyplot as plt
    import matplotlib.ticker as mtick

    ROOT = Path.cwd()
    RESULTS = ROOT / "results"
    plt.style.use("seaborn-v0_8-whitegrid")
    plt.rcParams["figure.figsize"] = (13, 5)
    plt.rcParams["axes.unicode_minus"] = False
    for font in ["Malgun Gothic", "AppleGothic", "DejaVu Sans"]:
        try:
            plt.rcParams["font.family"] = font
            break
        except Exception:
            pass

    pd.set_option("display.max_columns", 40)
    pd.set_option("display.width", 180)
    print("작업 폴더:", ROOT)
    print("자산:", ["KODEX200", "BOND", "GLD", "USO"])

## 2. 기존 모델 구현

    아래 숨김 셀은 거시데이터 로딩, Sparse Jump Model, soft 국면 확률, 비대칭 EWMA 공분산, CDaR/변동성 목표, drawdown guard, 거래비용 차감 walk-forward 백테스트를 포함한다.

In [ ]:
from __future__ import annotations

import json
import math
import sqlite3
import unicodedata
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import expit
from sklearn.metrics import balanced_accuracy_score, confusion_matrix

warnings.filterwarnings("ignore", category=FutureWarning)

ASSETS = ["KODEX200", "BOND", "GLD", "USO"]
ROOT = Path.cwd()
RAW_DIR = ROOT / "raw_data"
CACHE_DIR = ROOT / "cache"
RESULTS_DIR = ROOT / "results"
CACHE_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)


def get_path(directory: Path, filename: str) -> Path:
    target = unicodedata.normalize("NFC", filename)
    for path in directory.iterdir():
        if unicodedata.normalize("NFC", path.name) == target:
            return path
    raise FileNotFoundError(filename)


def rolling_zscore(series: pd.Series, window: int, clip: float = 3.0) -> pd.Series:
    mean = series.rolling(window, min_periods=window).mean()
    std = series.rolling(window, min_periods=window).std(ddof=1).replace(0, np.nan)
    return ((series - mean) / std).clip(-clip, clip)


def load_macro_data() -> tuple[pd.DataFrame, pd.DataFrame]:
    gdp = pd.read_excel(get_path(RAW_DIR, "GDP 성장률.xlsx"), index_col=0, skiprows=6)
    gdp.columns = ["QoQ", "YoY"]
    gdp.index = pd.PeriodIndex(gdp.index, freq="Q").asfreq("M", how="end").to_timestamp("M") + pd.offsets.MonthEnd(1)
    gdp = gdp.resample("ME").ffill()

    trade = pd.read_excel(get_path(RAW_DIR, "수출입 총괄_20260816.xlsx"), index_col=0, skiprows=4)
    trade = trade[["수출 금액", "수입금액"]].iloc[1:].copy()
    for col in trade.columns:
        trade[col] = trade[col].astype(str).str.replace(",", "", regex=False).astype(float)
    trade.index = pd.to_datetime(trade.index, format="%Y.%m") + pd.offsets.MonthEnd(1)
    trade["Export_YoY"] = trade["수출 금액"].pct_change(12) * 100

    bsi = pd.read_csv(get_path(RAW_DIR, "기업경기조사(전망).csv"), encoding="cp949")
    bsi = bsi[(bsi["업종코드별"] == "제 조 업") & (bsi["BSI코드별"] == "업황전망BSI 1)")]
    bsi = bsi.iloc[:, 2:4].copy()
    bsi["시점"] = bsi["시점"].str.replace("월", "", regex=False).str.replace(" ", "", regex=False)
    bsi["시점"] = pd.to_datetime(bsi["시점"], format="%Y.%m") + pd.offsets.MonthEnd(1)
    bsi = bsi.set_index("시점")
    bsi.columns = ["BSI"]

    cpi = pd.read_excel(get_path(RAW_DIR, "소비자물가 상승률.xlsx"), index_col=0, skiprows=6)
    cpi.columns = ["CPI_QoQ", "CPI_YoY"]
    cpi.index = pd.to_datetime(cpi.index, format="%Y-%m") + pd.offsets.MonthEnd(2)

    ppi = pd.read_excel(get_path(RAW_DIR, "생산자물가 상승률.xlsx"), index_col=0, skiprows=6)
    ppi.columns = ["PPI_QoQ", "PPI_YoY"]
    ppi.index = pd.to_datetime(ppi.index, format="%Y-%m") + pd.offsets.MonthEnd(2)

    prices = pd.read_excel(get_path(RAW_DIR, "수출입물가 상승률.xlsx"), index_col=0, skiprows=6)
    prices.columns = ["ExportPrice_YoY", "ImportPrice_YoY"]
    prices.index = pd.to_datetime(prices.index, format="%Y-%m") + pd.offsets.MonthEnd(2)

    core = pd.concat(
        {
            "GDP": rolling_zscore(gdp["YoY"], 72),
            "Export": rolling_zscore(trade["Export_YoY"], 36),
            "BSI": rolling_zscore(bsi["BSI"], 24),
            "CPI": rolling_zscore(cpi["CPI_YoY"], 36),
            "PPI": rolling_zscore(ppi["PPI_YoY"], 36),
            "ImportPrice": rolling_zscore(prices["ImportPrice_YoY"], 36),
        },
        axis=1,
    ).sort_index()

    # Levels describe the phase; 3-month changes help identify turning points.
    growth = core[["GDP", "Export", "BSI"]].copy()
    growth.columns = ["GDP_level", "Export_level", "BSI_level"]
    growth = pd.concat([growth, growth.diff(3).add_suffix("_d3")], axis=1)
    inflation = core[["CPI", "PPI", "ImportPrice"]].copy()
    inflation.columns = ["CPI_level", "PPI_level", "ImportPrice_level"]
    inflation = pd.concat([inflation, inflation.diff(3).add_suffix("_d3")], axis=1)
    features = pd.concat({"growth": growth, "inflation": inflation}, axis=1).dropna()
    return features, core


def download_market_cache(refresh: bool = False) -> pd.DataFrame:
    cache = CACHE_DIR / "market_daily.csv"
    if cache.exists() and not refresh:
        out = pd.read_csv(cache, parse_dates=["date"])
        return out

    import yfinance as yf

    rows: list[pd.DataFrame] = []
    for ticker, symbol in [("069500.KS", "KODEX200"), ("GLD", "GLD"), ("USO", "USO"), ("KRW=X", "USDKRW")]:
        data = yf.download(ticker, start="2000-01-01", auto_adjust=(symbol != "USDKRW"), progress=False, threads=False)
        if isinstance(data.columns, pd.MultiIndex):
            data.columns = data.columns.get_level_values(0)
        data = data.reset_index().rename(columns={"Date": "date", "Open": "open", "Close": "close"})
        data["date"] = pd.to_datetime(data["date"], utc=True).dt.tz_localize(None).dt.normalize()
        data["symbol"] = symbol
        rows.append(data[["date", "symbol", "open", "close"]])
    out = pd.concat(rows, ignore_index=True).dropna(subset=["date", "close"])
    out.to_csv(cache, index=False)
    return out


def load_monthly_asset_returns(refresh: bool = False) -> tuple[pd.DataFrame, pd.DataFrame]:
    market = download_market_cache(refresh)

    with sqlite3.connect(get_path(RAW_DIR, "compass.db")) as con:
        proxy = pd.read_sql(
            "select date, open, close from etf_prices where symbol = ? order by date",
            con,
            params=("1028",),
        )
    proxy["date"] = pd.to_datetime(proxy["date"])
    proxy[["open", "close"]] = proxy[["open", "close"]].apply(pd.to_numeric, errors="coerce")

    actual = market[market["symbol"] == "KODEX200"].copy().dropna(subset=["open"])
    # Yahoo contains a sparse early fragment followed by a long gap.  Use the
    # continuous KOSPI200 proxy through March 2009, then splice the ETF series.
    actual = actual[actual["date"] > pd.Timestamp("2009-03-31")]
    first_actual = actual["date"].min()
    actual_anchor = actual.loc[actual["date"] == first_actual, "open"].iloc[0]
    proxy_anchor = proxy.loc[proxy["date"] == first_actual, "open"]
    if proxy_anchor.empty:
        nearest = proxy.iloc[(proxy["date"] - first_actual).abs().argsort()[:1]]
        proxy_anchor_value = float(nearest["open"].iloc[0])
    else:
        proxy_anchor_value = float(proxy_anchor.iloc[0])
    proxy["open"] = proxy["open"] * float(actual_anchor) / proxy_anchor_value
    proxy["close"] = proxy["close"] * float(actual_anchor) / proxy_anchor_value
    proxy = proxy[proxy["date"] < first_actual]
    proxy["symbol"] = "KODEX200"
    kodex = pd.concat([proxy[["date", "symbol", "open", "close"]], actual], ignore_index=True)

    bond = pd.read_csv(get_path(RAW_DIR, "krx_bond_index.csv"), encoding="cp949")
    bond["date"] = pd.to_datetime(bond.iloc[:, 0])
    bond["open"] = bond.iloc[:, 1].astype(str).str.replace(",", "", regex=False).astype(float)
    bond["close"] = bond["open"]
    bond["symbol"] = "BOND"

    fx = market[market["symbol"] == "USDKRW"].set_index("date")["close"].sort_index()
    fx = fx.reindex(pd.date_range(fx.index.min(), fx.index.max(), freq="D")).ffill()

    first_open: dict[str, pd.Series] = {}
    trade_dates: dict[str, pd.Series] = {}
    for symbol, data in {
        "KODEX200": kodex,
        "BOND": bond,
        "GLD": market[market["symbol"] == "GLD"],
        "USO": market[market["symbol"] == "USO"],
    }.items():
        temp = data.dropna(subset=["open"]).sort_values("date").copy()
        temp["month"] = temp["date"].dt.to_period("M")
        first = temp.groupby("month", sort=True).first()
        value = first["open"].astype(float)
        if symbol in {"GLD", "USO"}:
            value = value * fx.reindex(pd.DatetimeIndex(first["date"]), method="ffill").to_numpy()
        first_open[symbol] = value
        trade_dates[symbol] = first["date"]

    levels = pd.concat(first_open, axis=1).sort_index()
    returns = levels.shift(-1).div(levels).sub(1.0).dropna(how="any")
    returns = returns[ASSETS]
    return returns, levels[ASSETS]


def _softmax(values: np.ndarray) -> np.ndarray:
    z = values - np.max(values)
    exp = np.exp(z)
    return exp / exp.sum()


class SparseJump2:
    """Small, transparent two-state sparse jump model.

    It alternates between sparse feature weighting/centroid estimation and a
    dynamic-programming state sequence with an explicit switching penalty.
    """

    def __init__(self, jump_penalty: float = 3.0, keep_features: int = 4, max_iter: int = 30):
        self.jump_penalty = float(jump_penalty)
        self.keep_features = int(keep_features)
        self.max_iter = int(max_iter)

    @staticmethod
    def _dp(dist: np.ndarray, jump: float) -> tuple[np.ndarray, np.ndarray]:
        n = len(dist)
        costs = np.zeros((n, 2))
        back = np.zeros((n, 2), dtype=int)
        costs[0] = dist[0]
        for t in range(1, n):
            for k in range(2):
                candidates = costs[t - 1] + jump * (np.arange(2) != k)
                back[t, k] = int(np.argmin(candidates))
                costs[t, k] = dist[t, k] + candidates[back[t, k]]
        states = np.zeros(n, dtype=int)
        states[-1] = int(np.argmin(costs[-1]))
        for t in range(n - 2, -1, -1):
            states[t] = back[t + 1, states[t + 1]]
        return states, costs

    def fit_predict_high(self, frame: pd.DataFrame) -> tuple[float, dict]:
        x_raw = frame.to_numpy(dtype=float)
        med = np.nanmedian(x_raw, axis=0)
        scale = np.nanpercentile(x_raw, 75, axis=0) - np.nanpercentile(x_raw, 25, axis=0)
        scale = np.where(scale < 0.15, np.nanstd(x_raw, axis=0), scale)
        scale = np.where(scale < 1e-6, 1.0, scale)
        x = np.clip((x_raw - med) / scale, -5, 5)

        score = np.nanmean(x[:, : min(3, x.shape[1])], axis=1)
        states = (score > np.nanmedian(score)).astype(int)
        weights = np.ones(x.shape[1]) / x.shape[1]
        for _ in range(self.max_iter):
            old = states.copy()
            centers = np.vstack([
                x[states == k].mean(axis=0) if np.any(states == k) else np.nanmean(x, axis=0)
                for k in range(2)
            ])
            within = np.vstack([
                np.nanvar(x[states == k], axis=0) if np.sum(states == k) > 1 else np.ones(x.shape[1])
                for k in range(2)
            ]).mean(axis=0)
            separation = (centers[1] - centers[0]) ** 2 / (within + 0.20)
            keep = np.argsort(separation)[-min(self.keep_features, len(separation)) :]
            weights = np.zeros_like(separation)
            weights[keep] = np.maximum(separation[keep], 1e-4)
            weights /= weights.sum()
            dist = np.stack([((x - centers[k]) ** 2 * weights).sum(axis=1) for k in range(2)], axis=1)
            states, costs = self._dp(dist, self.jump_penalty)
            if np.array_equal(states, old):
                break

        centers = np.vstack([x[states == k].mean(axis=0) for k in range(2)])
        high_state = int(np.argmax(centers[:, : min(3, x.shape[1])].mean(axis=1)))
        prev_state = int(states[-2]) if len(states) > 1 else int(states[-1])
        local_dist = np.array([((x[-1] - centers[k]) ** 2 * weights).sum() for k in range(2)])
        local_cost = local_dist + self.jump_penalty * 0.55 * (np.arange(2) != prev_state)
        probs = _softmax(-local_cost / 0.85)
        p_high = float(np.clip(probs[high_state], 0.03, 0.97))
        detail = {
            "p_high": p_high,
            "state": int(states[-1]),
            "high_state": high_state,
            "switches": int(np.sum(states[1:] != states[:-1])),
            "feature_weights": dict(zip(frame.columns, weights)),
        }
        return p_high, detail


def compute_regime_signals(features: pd.DataFrame, returns: pd.DataFrame, jump_penalty: float = 3.0, min_history: int = 24) -> pd.DataFrame:
    rows = []
    model = SparseJump2(jump_penalty=jump_penalty, keep_features=4)
    pg_prev = 0.5
    pi_prev = 0.5
    for target_month in returns.index:
        signal_month = target_month - 1
        hist = features.loc[: signal_month.to_timestamp("M")]
        if len(hist) < min_history:
            continue
        pg_sjm, gd = model.fit_predict_high(hist["growth"])
        pi_sjm, id_ = model.fit_predict_high(hist["inflation"])
        growth_now = float(hist["growth"].iloc[-1][["GDP_level", "Export_level", "BSI_level"]].mean())
        growth_mom = float(hist["growth"].iloc[-1][["GDP_level_d3", "Export_level_d3", "BSI_level_d3"]].mean())
        inflation_now = float(hist["inflation"].iloc[-1][["CPI_level", "PPI_level", "ImportPrice_level"]].mean())
        inflation_mom = float(hist["inflation"].iloc[-1][["CPI_level_d3", "PPI_level_d3", "ImportPrice_level_d3"]].mean())
        pg_composite = float(expit((growth_now + 0.20 * growth_mom) / 0.55))
        pi_composite = float(expit((inflation_now + 0.20 * inflation_mom) / 0.55))
        # The transparent composite is the primary forecast because the sample
        # is small; SJM contributes sparse selection and switch persistence.
        pg_raw = 0.10 * pg_sjm + 0.90 * pg_composite
        pi_raw = 0.10 * pi_sjm + 0.90 * pi_composite
        pg = 0.85 * pg_raw + 0.15 * pg_prev
        pi = 0.85 * pi_raw + 0.15 * pi_prev
        pg_prev, pi_prev = pg, pi
        probs = {
            "Goldilocks": pg * (1 - pi),
            "Overheating": pg * pi,
            "Slowdown": (1 - pg) * (1 - pi),
            "Stagflation": (1 - pg) * pi,
        }
        regime = max(probs, key=probs.get)
        rows.append(
            {
                "target_month": target_month,
                "signal_month": signal_month,
                "p_growth_high": pg,
                "p_inflation_high": pi,
                "p_growth_sjm": pg_sjm,
                "p_inflation_sjm": pi_sjm,
                "growth_composite": growth_now,
                "inflation_composite": inflation_now,
                "regime": regime,
                **{f"p_{k}": v for k, v in probs.items()},
                "growth_switches": gd["switches"],
                "inflation_switches": id_["switches"],
                "growth_features": json.dumps(gd["feature_weights"], ensure_ascii=False),
                "inflation_features": json.dumps(id_["feature_weights"], ensure_ascii=False),
            }
        )
    return pd.DataFrame(rows).set_index("target_month")


REGIME_ANCHORS = pd.DataFrame(
    {
        "Goldilocks": [0.58, 0.22, 0.15, 0.05],
        "Overheating": [0.30, 0.12, 0.23, 0.35],
        "Slowdown": [0.12, 0.66, 0.20, 0.02],
        "Stagflation": [0.08, 0.24, 0.50, 0.18],
    },
    index=ASSETS,
).T
DEFENSIVE = np.array([0.05, 0.72, 0.23, 0.00])
STRATEGIC = np.array([0.20, 0.45, 0.30, 0.05])


def soft_anchor(signal: pd.Series) -> np.ndarray:
    p = np.array([signal[f"p_{r}"] for r in REGIME_ANCHORS.index])
    return p @ REGIME_ANCHORS.to_numpy()


def ewma_cov(history: pd.DataFrame, half_life: float = 12.0, leverage: float = 1.0) -> np.ndarray:
    x = history[ASSETS].to_numpy(dtype=float)
    if len(x) < 12:
        return np.cov(x, rowvar=False) + np.eye(len(ASSETS)) * 1e-6
    alpha = 1 - math.exp(math.log(0.5) / half_life)
    cov = np.cov(x[: min(24, len(x))], rowvar=False)
    mean = np.nanmean(x, axis=0)
    for row in x:
        shock = row - mean
        multiplier = 1.0 + leverage * min(max(-row[0], 0.0) / 0.08, 1.5)
        cov = (1 - alpha) * cov + alpha * multiplier * np.outer(shock, shock)
    return cov + np.eye(len(ASSETS)) * 1e-7


def cdar(returns: np.ndarray, alpha: float = 0.90) -> float:
    wealth = np.cumprod(1 + returns)
    dd = wealth / np.maximum.accumulate(np.r_[1.0, wealth])[-len(wealth):] - 1.0
    k = max(1, int(math.ceil((1 - alpha) * len(dd))))
    return float(np.mean(np.sort(dd)[:k]))


@dataclass
class StrategyConfig:
    name: str = "Proposed"
    target_vol: float = 0.08
    half_life: float = 12.0
    invvol_tilt: float = 0.35
    return_reward: float = 1.15
    vol_penalty: float = 0.18
    cdar_penalty: float = 0.25
    turnover_penalty: float = 0.05
    tracking_penalty: float = 0.32
    max_cdar: float = 0.16
    drawdown_guard: float = 0.75
    regime_strength: float = 0.75
    use_regime: bool = True
    use_risk_control: bool = True


def controlled_weights(
    signal: pd.Series,
    history: pd.DataFrame,
    pretrade: np.ndarray,
    current_dd: float,
    cfg: StrategyConfig,
) -> np.ndarray:
    anchor = soft_anchor(signal) if cfg.use_regime else STRATEGIC.copy()
    anchor = cfg.regime_strength * anchor + (1 - cfg.regime_strength) * STRATEGIC
    if not cfg.use_risk_control or len(history) < 24:
        return anchor

    cov = ewma_cov(history.tail(84), cfg.half_life, leverage=1.0)
    vols = np.sqrt(np.diag(cov)).clip(0.005, None)
    tilted = anchor * (np.median(vols) / vols) ** cfg.invvol_tilt
    tilted = tilted / tilted.sum()
    prior = 0.55 * anchor + 0.45 * tilted

    hist = history.tail(84)[ASSETS]
    long_mu = history[ASSETS].expanding(min_periods=24).mean().iloc[-1].to_numpy()
    recent_mu = hist.ewm(halflife=24, adjust=False).mean().iloc[-1].to_numpy()
    mu = 0.80 * long_mu + 0.20 * recent_mu
    mu = np.clip(mu, -0.006, 0.015)
    target = cfg.target_vol * (0.86 + 0.20 * float(signal["p_growth_high"]))

    def objective(w: np.ndarray) -> float:
        ann_return = 12 * float(w @ mu)
        ann_vol = math.sqrt(max(float(w @ cov @ w), 0.0) * 12)
        path_cdar = abs(cdar(hist.to_numpy() @ w, 0.90))
        turnover = 0.5 * np.sum(np.sqrt((w - pretrade) ** 2 + 1e-6))
        tracking = float(np.sum((w - prior) ** 2))
        return (
            -cfg.return_reward * ann_return
            + cfg.vol_penalty * ann_vol
            + cfg.cdar_penalty * path_cdar
            + cfg.turnover_penalty * turnover
            + cfg.tracking_penalty * tracking
        )

    constraints = [
        {"type": "eq", "fun": lambda w: np.sum(w) - 1.0},
        {"type": "ineq", "fun": lambda w: target - math.sqrt(max(float(w @ cov @ w), 0.0) * 12)},
        {"type": "ineq", "fun": lambda w: cfg.max_cdar + cdar(hist.to_numpy() @ w, 0.90)},
    ]
    bounds = [(0.02, 0.68), (0.05, 0.88), (0.02, 0.62), (0.0, 0.38)]
    result = minimize(objective, prior, method="SLSQP", bounds=bounds, constraints=constraints, options={"maxiter": 80, "ftol": 1e-8})
    w = result.x if result.success and np.isfinite(result.x).all() else prior
    w = np.clip(w, 0, None)
    w /= w.sum()

    # MPC-style state-dependent risk aversion: react to realized drawdown, but
    # retain a floor in risky assets so recovery participation is not lost.
    if current_dd < -0.05:
        severity = min(max((-current_dd - 0.05) / 0.12, 0.0), 1.0)
        blend = cfg.drawdown_guard * (0.25 + 0.50 * severity)
        w = (1 - blend) * w + blend * DEFENSIVE
    return w / w.sum()


def hard_regime_weights(signal: pd.Series) -> np.ndarray:
    mapping = {
        "Goldilocks": np.array([1.0, 0.0, 0.0, 0.0]),
        "Overheating": np.array([0.0, 0.0, 0.0, 1.0]),
        "Slowdown": np.array([0.6, 0.4, 0.0, 0.0]),
        "Stagflation": np.array([0.0, 0.0, 1.0, 0.0]),
    }
    return mapping[signal["regime"]]


def run_backtest(
    returns: pd.DataFrame,
    signals: pd.DataFrame,
    cfg: StrategyConfig,
    mode: str = "proposed",
    start: str | None = None,
    end: str | None = None,
    cost_multiplier: float = 1.0,
) -> pd.DataFrame:
    months = signals.index.intersection(returns.index)
    if start:
        months = months[months >= pd.Period(start, "M")]
    if end:
        months = months[months <= pd.Period(end, "M")]
    rows = []
    pretrade = np.zeros(4)
    first_trade = True
    nav = 1.0
    peak = 1.0
    for month in months:
        signal = signals.loc[month]
        history = returns.loc[returns.index < month]
        current_dd = nav / peak - 1.0
        if mode == "proposed":
            w = controlled_weights(signal, history, pretrade, current_dd, cfg)
        elif mode == "soft":
            w = soft_anchor(signal)
        elif mode == "hard":
            w = hard_regime_weights(signal)
        elif mode == "equal":
            w = np.full(4, 0.25)
        elif mode == "static_defensive":
            w = np.array([0.20, 0.45, 0.30, 0.05])
        elif mode == "kodex":
            w = np.array([1.0, 0.0, 0.0, 0.0])
        else:
            raise ValueError(mode)

        delta = w - pretrade
        turnover = np.abs(delta).sum() if first_trade else 0.5 * np.abs(delta).sum()
        trade_cost = np.abs(delta).sum() * 0.0015 * cost_multiplier
        fx_cost = abs((w[2] + w[3]) - (pretrade[2] + pretrade[3])) * 0.0005 * cost_multiplier
        gross_return = float(w @ returns.loc[month, ASSETS].to_numpy())
        net_return = gross_return - trade_cost - fx_cost
        nav *= 1 + net_return
        peak = max(peak, nav)
        end_w = w * (1 + returns.loc[month, ASSETS].to_numpy()) / (1 + gross_return)
        rows.append(
            {
                "month": month,
                "signal_month": signal["signal_month"],
                "regime": signal["regime"],
                "p_growth_high": signal["p_growth_high"],
                "p_inflation_high": signal["p_inflation_high"],
                "gross_return": gross_return,
                "return": net_return,
                "turnover": turnover,
                "trade_cost": trade_cost,
                "fx_cost": fx_cost,
                "nav": nav,
                "drawdown": nav / peak - 1,
                **{f"w_{a}": w[i] for i, a in enumerate(ASSETS)},
            }
        )
        pretrade = end_w
        first_trade = False
    return pd.DataFrame(rows).set_index("month")


def performance_summary(returns: pd.Series) -> pd.Series:
    r = pd.Series(returns).dropna()
    wealth = (1 + r).cumprod()
    years = len(r) / 12
    cagr = wealth.iloc[-1] ** (1 / years) - 1 if years > 0 else np.nan
    vol = r.std(ddof=1) * math.sqrt(12)
    sharpe = r.mean() / r.std(ddof=1) * math.sqrt(12) if r.std(ddof=1) > 0 else np.nan
    dd = wealth / wealth.cummax() - 1
    mdd = dd.min()
    calmar = cagr / abs(mdd) if mdd < 0 else np.nan
    downside = np.sqrt(np.mean(np.minimum(r, 0) ** 2)) * math.sqrt(12)
    sortino = r.mean() * 12 / downside if downside > 0 else np.nan
    return pd.Series(
        {
            "Months": len(r),
            "CAGR": cagr,
            "Volatility": vol,
            "Sharpe": sharpe,
            "Sortino": sortino,
            "MDD": mdd,
            "Calmar": calmar,
            "FinalMultiple": wealth.iloc[-1],
            "PositiveMonths": (r > 0).mean(),
        }
    )


def evaluate_regimes(signals: pd.DataFrame, core: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    composite = pd.DataFrame(
        {
            "growth_realized": core[["GDP", "Export", "BSI"]].mean(axis=1),
            "inflation_realized": core[["CPI", "PPI", "ImportPrice"]].mean(axis=1),
        }
    )
    # Ex-post target: average of the next three published monthly readings.
    future = pd.concat([composite.shift(-k) for k in (1, 2, 3)], axis=1)
    future.columns = pd.MultiIndex.from_product([[1, 2, 3], composite.columns])
    target = pd.DataFrame(index=composite.index)
    target["growth_high_realized"] = future.xs("growth_realized", axis=1, level=1).mean(axis=1) >= 0
    target["inflation_high_realized"] = future.xs("inflation_realized", axis=1, level=1).mean(axis=1) >= 0
    pred = signals.copy()
    pred.index = pred["signal_month"].apply(lambda x: x.to_timestamp("M"))
    joined = pred.join(target, how="inner").dropna(subset=["growth_high_realized", "inflation_high_realized"])
    joined["growth_pred"] = joined["p_growth_high"] >= 0.5
    joined["inflation_pred"] = joined["p_inflation_high"] >= 0.5
    joined["quadrant_hit"] = (joined["growth_pred"] == joined["growth_high_realized"]) & (joined["inflation_pred"] == joined["inflation_high_realized"])
    metrics = {
        "growth_balanced_accuracy": balanced_accuracy_score(joined["growth_high_realized"], joined["growth_pred"]),
        "inflation_balanced_accuracy": balanced_accuracy_score(joined["inflation_high_realized"], joined["inflation_pred"]),
        "quadrant_accuracy": float(joined["quadrant_hit"].mean()),
        "n_months": len(joined),
        "growth_confusion": confusion_matrix(joined["growth_high_realized"], joined["growth_pred"]).tolist(),
        "inflation_confusion": confusion_matrix(joined["inflation_high_realized"], joined["inflation_pred"]).tolist(),
    }
    return joined, metrics



## 3. CAGR 개선 오버레이 구현

    경제적 논리는 단순하다.

    1. 최근 12개월 수익을 변동성으로 나눈 추세 점수가 양수이고 1·3개월 수익도 양수인 위험자산만 선택한다.
    2. 채권에서 최대 10%p를 꺼내 상위 두 자산에 배분한다. 추정 연 변동성이 10%를 넘으면 이 비중을 축소한다.
    3. KODEX200·USO의 3·6개월 추세가 모두 음수면 해당 비중의 25%를 채권으로 이동한다. 금은 위기 분산자산이므로 이 추세 브레이크에서 제외한다.
    4. 포트폴리오 낙폭이 -1.5%를 넘으면 방어 비중을 점진적으로 높이고, -2.5% 이하면 추가 위험 확대를 중단한다.

    모든 판단은 해당 투자월 이전에 관측 가능한 수익률과 당시까지의 포트폴리오 낙폭만 사용한다.

In [ ]:
from __future__ import annotations

import itertools
import json
import math
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd

from strategies.core.regime_research import (
    ASSETS,
    StrategyConfig,
    compute_regime_signals,
    controlled_weights,
    ewma_cov,
    load_macro_data,
    load_monthly_asset_returns,
    performance_summary,
    run_backtest,
)


ROOT = Path.cwd()
RESULTS = ROOT / "results"
RISKY = ["KODEX200", "GLD", "USO"]
CAPS = {"KODEX200": 0.62, "GLD": 0.60, "USO": 0.22}


@dataclass(frozen=True)
class AcceleratorConfig:
    sleeve: float
    vol_cap: float
    trend_window: int
    top_k: int
    brake_fraction: float = 0.0
    early_guard_start: float = -0.02
    early_guard_strength: float = 0.0
    dd_gate: float = -0.025
    bond_floor: float = 0.18

    @property
    def name(self) -> str:
        return (
            f"sl{self.sleeve:.2f}_vc{self.vol_cap:.3f}_tw{self.trend_window}_k{self.top_k}"
            f"_bf{self.brake_fraction:.2f}_egs{abs(self.early_guard_start):.3f}_eg{self.early_guard_strength:.2f}"
        )


def add_accelerator(
    base: np.ndarray,
    history: pd.DataFrame,
    current_dd: float,
    acfg: AcceleratorConfig,
) -> tuple[np.ndarray, str, float, float]:
    if len(history) < max(24, acfg.trend_window):
        return base, "OFF_HISTORY", 0.0, 0.0

    trailing = history.tail(acfg.trend_window)[ASSETS]
    mom = (1 + trailing).prod() - 1
    mom1 = history.iloc[-1][ASSETS]
    mom3 = (1 + history.tail(3)[ASSETS]).prod() - 1
    ann_vol = trailing.std(ddof=1) * math.sqrt(12)
    score = mom / ann_vol.clip(0.04)

    # Symmetric trend brake: the accelerator earns the right to take more risk
    # only if existing equity/oil risk is cut when both medium and short trends
    # are negative.  Gold is retained as the crisis diversifier.
    working = base.copy()
    brake_used = 0.0
    mom6 = (1 + history.tail(6)[ASSETS]).prod() - 1
    for asset in ["KODEX200", "USO"]:
        idx = ASSETS.index(asset)
        if mom3[asset] < 0 and mom6[asset] < 0:
            cut = acfg.brake_fraction * working[idx]
            working[idx] -= cut
            working[ASSETS.index("BOND")] += cut
            brake_used += cut

    if current_dd <= acfg.dd_gate:
        return working / working.sum(), "BRAKE_DD" if brake_used else "OFF_DD", 0.0, brake_used

    eligible = [a for a in RISKY if mom[a] > 0 and mom3[a] > 0 and mom1[a] > 0 and score[a] > 0]
    if not eligible:
        return working / working.sum(), "BRAKE" if brake_used else "OFF_TREND", 0.0, brake_used

    selected = sorted(eligible, key=lambda a: score[a], reverse=True)[: acfg.top_k]
    raw_strength = float(np.clip(max(mom[a] for a in selected) / 0.12, 0.25, 1.0))
    available = max(float(working[ASSETS.index("BOND")]) - acfg.bond_floor, 0.0)
    dd_scale = float(np.clip((current_dd - acfg.dd_gate) / (0.0 - acfg.dd_gate), 0.0, 1.0))
    desired = min(acfg.sleeve * raw_strength * dd_scale, available)
    if desired <= 1e-8:
        return working / working.sum(), "BRAKE_FLOOR" if brake_used else "OFF_FLOOR", 0.0, brake_used

    positive_scores = np.array([max(float(score[a]), 1e-6) for a in selected])
    shares = positive_scores / positive_scores.sum()

    def candidate(amount: float) -> np.ndarray:
        w = working.copy()
        w[ASSETS.index("BOND")] -= amount
        unallocated = amount
        for asset, share in zip(selected, shares):
            idx = ASSETS.index(asset)
            add = min(amount * float(share), CAPS[asset] - w[idx])
            w[idx] += max(add, 0.0)
            unallocated -= max(add, 0.0)
        w[ASSETS.index("BOND")] += max(unallocated, 0.0)
        return w / w.sum()

    cov = ewma_cov(history.tail(84), half_life=12.0, leverage=1.0)
    proposed = candidate(desired)
    proposed_vol = math.sqrt(max(float(proposed @ cov @ proposed), 0.0) * 12)
    if proposed_vol > acfg.vol_cap:
        lo, hi = 0.0, desired
        for _ in range(30):
            mid = 0.5 * (lo + hi)
            w_mid = candidate(mid)
            vol_mid = math.sqrt(max(float(w_mid @ cov @ w_mid), 0.0) * 12)
            if vol_mid <= acfg.vol_cap:
                lo = mid
            else:
                hi = mid
        desired = lo
        proposed = candidate(desired)

    if desired <= 1e-5:
        return working / working.sum(), "BRAKE_VOL" if brake_used else "OFF_VOL", 0.0, brake_used
    label = "+".join(selected) + ("|BRAKE" if brake_used else "")
    return proposed, label, desired, brake_used


def run_accelerated(
    returns: pd.DataFrame,
    signals: pd.DataFrame,
    base_cfg: StrategyConfig,
    acfg: AcceleratorConfig,
    start: str | None = None,
    end: str | None = None,
    cost_multiplier: float = 1.0,
) -> pd.DataFrame:
    months = signals.index.intersection(returns.index)
    if start:
        months = months[months >= pd.Period(start, "M")]
    if end:
        months = months[months <= pd.Period(end, "M")]

    rows = []
    pretrade = np.zeros(len(ASSETS))
    nav = 1.0
    peak = 1.0
    first_trade = True
    for month in months:
        signal = signals.loc[month]
        history = returns.loc[returns.index < month]
        current_dd = nav / peak - 1.0
        base = controlled_weights(signal, history, pretrade, current_dd, base_cfg)
        w, accelerator_asset, sleeve_used, brake_used = add_accelerator(base, history, current_dd, acfg)

        early_guard_used = 0.0
        if current_dd < acfg.early_guard_start and acfg.early_guard_strength > 0:
            severity = float(np.clip((acfg.early_guard_start - current_dd) / 0.05, 0.0, 1.0))
            early_guard_used = acfg.early_guard_strength * (0.30 + 0.70 * severity)
            w = (1 - early_guard_used) * w + early_guard_used * np.array([0.05, 0.72, 0.23, 0.00])
            w /= w.sum()

        delta = w - pretrade
        turnover = np.abs(delta).sum() if first_trade else 0.5 * np.abs(delta).sum()
        trade_cost = np.abs(delta).sum() * 0.0015 * cost_multiplier
        fx_cost = abs((w[2] + w[3]) - (pretrade[2] + pretrade[3])) * 0.0005 * cost_multiplier
        asset_r = returns.loc[month, ASSETS].to_numpy()
        gross_return = float(w @ asset_r)
        net_return = gross_return - trade_cost - fx_cost
        nav *= 1 + net_return
        peak = max(peak, nav)
        end_w = w * (1 + asset_r) / (1 + gross_return)
        rows.append({
            "month": month,
            "return": net_return,
            "gross_return": gross_return,
            "nav": nav,
            "drawdown": nav / peak - 1,
            "turnover": turnover,
            "trade_cost": trade_cost,
            "fx_cost": fx_cost,
            "regime": signal["regime"],
            "accelerator_asset": accelerator_asset,
            "sleeve_used": sleeve_used,
            "brake_used": brake_used,
            "early_guard_used": early_guard_used,
            **{f"w_{asset}": w[i] for i, asset in enumerate(ASSETS)},
            **{f"base_w_{asset}": base[i] for i, asset in enumerate(ASSETS)},
        })
        pretrade = end_w
        first_trade = False
    return pd.DataFrame(rows).set_index("month")



## 4. 데이터 로딩과 완전 재계산

In [ ]:
features, core = load_macro_data()
    asset_returns, asset_levels = load_monthly_asset_returns(refresh=False)
    signals = compute_regime_signals(features, asset_returns)

    base_cfg = StrategyConfig()
    with (RESULTS / "cagr_accelerator_winner.json").open(encoding="utf-8") as f:
        accelerator_cfg = AcceleratorConfig(**json.load(f))

    baseline_full = run_backtest(asset_returns, signals, base_cfg)
    enhanced_full = run_accelerated(asset_returns, signals, base_cfg, accelerator_cfg)

    data_audit = pd.DataFrame({
        "시작": [features.index.min(), asset_returns.index.min(), signals.index.min()],
        "종료": [features.index.max(), asset_returns.index.max(), signals.index.max()],
        "관측치": [len(features), len(asset_returns), len(signals)],
    }, index=["거시 입력변수", "공통 자산수익률", "투자가능 신호"])
    display(data_audit)
    display(Markdown("### 선택된 전술 구성"))
    display(pd.Series(asdict(accelerator_cfg), name="Locked value").to_frame())

## 5. 핵심 결과 — 기존 모델과 개선안

In [ ]:
period_specs = [
        ("캘리브레이션 2007-2017", None, "2017-12"),
        ("잠금 테스트 2018+", "2018-01", None),
        ("전체 2007-2026", None, None),
    ]
    comparison_rows = []
    period_backtests = {}
    for period, start, end in period_specs:
        base_bt = run_backtest(asset_returns, signals, base_cfg, start=start, end=end)
        enhanced_bt = run_accelerated(asset_returns, signals, base_cfg, accelerator_cfg, start=start, end=end)
        period_backtests[(period, "기존")] = base_bt
        period_backtests[(period, "개선")] = enhanced_bt
        for strategy, bt in [("기존", base_bt), ("개선", enhanced_bt)]:
            m = performance_summary(bt["return"])
            comparison_rows.append({"구간": period, "전략": strategy, **m.to_dict(), "평균 월회전율": bt["turnover"].mean()})

    comparison = pd.DataFrame(comparison_rows)
    key_results = comparison[["구간", "전략", "CAGR", "Volatility", "Sharpe", "MDD", "Calmar", "평균 월회전율"]]
    display(key_results.style.format({
        "CAGR": "{:.2%}", "Volatility": "{:.2%}", "Sharpe": "{:.3f}",
        "MDD": "{:.2%}", "Calmar": "{:.3f}", "평균 월회전율": "{:.2%}",
    }).highlight_max(subset=["CAGR", "Sharpe", "Calmar"], color="#d9f2df"))

    base_m = performance_summary(baseline_full["return"])
    enh_m = performance_summary(enhanced_full["return"])
    delta = pd.Series({
        "CAGR 개선폭": enh_m["CAGR"] - base_m["CAGR"],
        "Sharpe 개선폭": enh_m["Sharpe"] - base_m["Sharpe"],
        "MDD 개선폭(양수=개선)": enh_m["MDD"] - base_m["MDD"],
        "Calmar 개선폭": enh_m["Calmar"] - base_m["Calmar"],
    })
    display(delta.to_frame("전체기간 차이").style.format({"전체기간 차이": "{:.3%}"}))

In [ ]:
idx = enhanced_full.index.to_timestamp()
    base_wealth = (1 + baseline_full["return"]).cumprod()
    enh_wealth = (1 + enhanced_full["return"]).cumprod()
    base_dd = base_wealth / base_wealth.cummax() - 1
    enh_dd = enh_wealth / enh_wealth.cummax() - 1

    fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
    axes[0].plot(idx, base_wealth, label="기존 모델", color="#7f8c8d", lw=1.7)
    axes[0].plot(idx, enh_wealth, label="CAGR 개선안", color="#0b63ce", lw=2.2)
    axes[0].set_yscale("log")
    axes[0].set_title("누적자산 비교 (로그축, 비용 차감 후)")
    axes[0].set_ylabel("1원의 성장")
    axes[0].legend()
    axes[1].plot(idx, base_dd, label="기존 모델", color="#7f8c8d", lw=1.5)
    axes[1].plot(idx, enh_dd, label="CAGR 개선안", color="#0b63ce", lw=1.9)
    axes[1].axhline(-0.10, color="#c0392b", ls="--", lw=1, label="-10% 기준")
    axes[1].set_title("Drawdown 비교")
    axes[1].set_ylabel("Drawdown")
    axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    axes[1].legend(ncol=3)
    plt.tight_layout()
    plt.show()

## 6. 비중 변화와 오버레이 작동 방식

In [ ]:
weight_cols = [f"w_{a}" for a in ASSETS]
    weights = enhanced_full[weight_cols].copy()
    weights.columns = ASSETS
    weights.index = weights.index.to_timestamp()

    fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
    weights.plot.area(ax=axes[0], color=["#1479ff", "#6c7a89", "#e5b94e", "#c85a3d"], alpha=0.88)
    axes[0].set_ylim(0, 1)
    axes[0].set_title("CAGR 개선안의 월별 목표비중")
    axes[0].set_ylabel("비중")
    axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    axes[0].legend(ncol=4, loc="upper center")

    overlay = enhanced_full[["sleeve_used", "brake_used", "early_guard_used"]].copy()
    overlay.index = overlay.index.to_timestamp()
    overlay.plot(ax=axes[1], color=["#0b63ce", "#c0392b", "#16a085"], lw=1.5)
    axes[1].set_title("전술 위험확대·추세 브레이크·조기 방어 강도")
    axes[1].set_ylabel("비중/강도")
    axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    axes[1].legend(["전술 sleeve", "추세 brake", "early guard"], ncol=3)
    plt.tight_layout()
    plt.show()

    base_weights = baseline_full[weight_cols].mean()
    enh_weights = enhanced_full[weight_cols].mean()
    weight_summary = pd.DataFrame({"기존 평균": base_weights.values, "개선 평균": enh_weights.values}, index=ASSETS)
    weight_summary["차이"] = weight_summary["개선 평균"] - weight_summary["기존 평균"]
    display(weight_summary.style.format("{:.1%}"))
    display(pd.Series({
        "평균 전술 sleeve": enhanced_full["sleeve_used"].mean(),
        "최대 전술 sleeve": enhanced_full["sleeve_used"].max(),
        "평균 추세 brake": enhanced_full["brake_used"].mean(),
        "평균 early guard": enhanced_full["early_guard_used"].mean(),
    }, name="사용 강도").to_frame().style.format("{:.2%}"))

## 7. 후보 선택 감사 — 공격적 후보를 왜 버렸는가

In [ ]:
calibration_grid = pd.read_csv(RESULTS / "cagr_accelerator_calibration.csv")
    calibration_grid["엄격 관문 통과"] = calibration_grid["MDDPass"] & calibration_grid["SharpePass"]
    audit_summary = pd.Series({
        "전체 후보 수": len(calibration_grid),
        "엄격 관문 통과 수": int(calibration_grid["엄격 관문 통과"].sum()),
        "15~20% 전술 후보 수": int((calibration_grid["sleeve"] >= 0.15).sum()),
        "15~20% 중 통과 수": int(((calibration_grid["sleeve"] >= 0.15) & calibration_grid["엄격 관문 통과"]).sum()),
    })
    display(audit_summary.to_frame("개수"))

    show_cols = ["name", "sleeve", "CAGR", "Sharpe", "MDD", "Calmar", "AvgTurnover", "엄격 관문 통과"]
    top_candidates = calibration_grid.sort_values("CAGR", ascending=False)[show_cols].head(12)
    display(top_candidates.style.format({
        "sleeve": "{:.0%}", "CAGR": "{:.2%}", "Sharpe": "{:.3f}", "MDD": "{:.2%}",
        "Calmar": "{:.3f}", "AvgTurnover": "{:.2%}",
    }))
    display(Markdown("**판정:** 공격적 후보는 CAGR이 높더라도 캘리브레이션 MDD가 기존보다 나빠져 채택하지 않았다."))

## 8. 하위기간·비용·위기 구간 견고성

In [ ]:
subperiod = pd.read_csv(RESULTS / "cagr_accelerator_subperiods.csv")
    subperiod_view = subperiod[["Period", "Strategy", "CAGR", "Sharpe", "MDD", "Calmar", "AvgTurnover"]]
    display(Markdown("### 하위기간"))
    display(subperiod_view.style.format({
        "CAGR": "{:.2%}", "Sharpe": "{:.3f}", "MDD": "{:.2%}", "Calmar": "{:.3f}", "AvgTurnover": "{:.2%}",
    }))

    costs = pd.read_csv(RESULTS / "cagr_accelerator_cost_sensitivity.csv")
    display(Markdown("### 거래·환전비용 민감도"))
    display(costs[["Period", "Strategy", "CAGR", "Sharpe", "MDD", "Calmar", "AnnualCost"]].style.format({
        "CAGR": "{:.2%}", "Sharpe": "{:.3f}", "MDD": "{:.2%}", "Calmar": "{:.3f}", "AnnualCost": "{:.2%}",
    }))

    episodes = pd.read_csv(RESULTS / "cagr_accelerator_stress_episodes.csv")
    display(Markdown("### 위기 에피소드"))
    display(episodes.style.format({"Return": "{:.2%}", "MDD": "{:.2%}", "Sharpe": "{:.3f}"}))

## 9. 쌍체 블록 부트스트랩

In [ ]:
bootstrap_ci = pd.read_csv(RESULTS / "cagr_accelerator_bootstrap_ci.csv", index_col=0)
    with (RESULTS / "cagr_accelerator_validation.json").open(encoding="utf-8") as f:
        validation = json.load(f)
    bootstrap_prob = pd.Series(validation["bootstrap_probabilities"], name="확률")

    display(Markdown("### 기존 대비 개선폭의 12개월 블록 부트스트랩 (3,000회)"))
    display(bootstrap_ci.style.format({
        "DeltaCAGR": "{:.2%}", "DeltaSharpe": "{:.3f}", "DeltaMDD": "{:.2%}", "DeltaCalmar": "{:.3f}",
    }))
    display(bootstrap_prob.to_frame().style.format("{:.1%}"))
    display(Markdown("양의 DeltaMDD는 개선안의 낙폭이 기존보다 덜 심하다는 뜻이다. 이 검정은 두 전략에 같은 재표본 경로를 적용해 개선분 자체의 불확실성을 측정한다."))

## 10. 자동 검증

In [ ]:
locked_base = period_backtests[("잠금 테스트 2018+", "기존")]
    locked_enh = period_backtests[("잠금 테스트 2018+", "개선")]
    locked_base_m = performance_summary(locked_base["return"])
    locked_enh_m = performance_summary(locked_enh["return"])

    assert signals.index.is_monotonic_increasing and signals.index.is_unique
    assert (signals["signal_month"] < signals.index).all(), "신호월은 투자월보다 앞서야 함"
    assert np.isfinite(enhanced_full["return"]).all()
    assert (enhanced_full[weight_cols].sum(axis=1).sub(1).abs() < 1e-8).all()
    assert (enhanced_full[weight_cols] >= -1e-12).all().all()
    assert (enhanced_full[["trade_cost", "fx_cost"]] >= 0).all().all()
    assert enh_m["CAGR"] > base_m["CAGR"], "전체기간 CAGR이 개선되지 않음"
    assert enh_m["MDD"] >= base_m["MDD"], "전체기간 MDD가 악화됨"
    assert locked_enh_m["CAGR"] > locked_base_m["CAGR"], "잠금구간 CAGR이 개선되지 않음"
    assert locked_enh_m["MDD"] >= locked_base_m["MDD"], "잠금구간 MDD가 악화됨"
    assert locked_enh_m["Sharpe"] > 1.0
    assert enhanced_full["sleeve_used"].max() <= accelerator_cfg.sleeve + 1e-10
    print("시점·비중·비용·CAGR·MDD·잠금 테스트 검증을 모두 통과했습니다.")

## 11. 결론

    - **전체기간:** CAGR은 약 **8.12% → 8.98%**, Sharpe는 **1.144 → 1.169**, MDD는 <strong>-8.93% → -8.90%</strong>로 개선됐다.
    - **잠금 테스트(2018+):** CAGR은 **9.02% → 10.54%**, Sharpe는 **1.183 → 1.247**, MDD는 <strong>-8.05% → -7.65%</strong>로 개선됐다.
    - **경제적 설명:** 경기·물가 국면이 기본 자산배분을 결정하고, 가격 추세는 위험을 새로 예측하는 모델이 아니라 채권 비중을 제한적으로 이동시키는 확인 장치다. 하락 추세·변동성·낙폭은 대칭적으로 위험을 다시 줄인다.
    - **견고성:** 12개월 쌍체 블록 부트스트랩에서 CAGR 개선 확률은 약 98%, Calmar 개선 확률은 약 93%였다. 비용을 2배 적용해도 개선안의 CAGR·Sharpe·Calmar가 기존보다 높았다.

    ### 현실적인 한계

    엄격한 MDD 비악화와 무레버리지·4개 자산 제약을 동시에 유지하면 전체기간 CAGR 10% 이상을 안정적으로 만들지는 못했다. 15~20% 전술 비중은 수익을 더 높였지만 MDD 관문을 통과하지 못했으므로 제외했다. 더 높은 목표에는 레버리지, 신규 수익원, 또는 더 큰 허용 MDD 중 하나가 필요하며, 이는 현재 요청 범위를 벗어난다.

    GDP·물가의 실시간 빈티지 부재, 2009-03 이전 KODEX200 프록시, 세금·시장충격 미반영 등 기존 검증판의 한계도 그대로 남는다.